# 00 — Setup smoke test

**Purpose.** Phase 1, task 3: load Gemma-2-2B via TransformerLens and verify a forward pass on a trivial prompt. This notebook is the env-proof step before `01_residual_stream.ipynb` does the real residual-stream work — if any cell here fails, fix it before moving on.

**What this verifies.**
1. The Python env can import `torch`, `transformer_lens`, `numpy`, `matplotlib`, `seaborn` (everything `01_residual_stream.ipynb` will need).
2. HuggingFace auth resolves a gated `google/gemma-2-2b` access.
3. `HookedTransformer.from_pretrained('gemma-2-2b')` downloads + loads weights on the local device.
4. A trivial forward pass on a two-shot Q/A prompt for *capital of France* picks `' Paris'` as the argmax next token.
5. The hook namespace contains the residual-stream hooks the next notebook depends on (`blocks.{i}.hook_resid_pre`).

**Citations.** Residual-stream framing follows Elhage et al. (2021), *A Mathematical Framework for Transformer Circuits* — `zotero://select/library/items/53EFBS8T`. See `vault/03-areas/research/mech-interp-overview.md` § Mechanism for the recipe this codebase implements.

**Prereq if a cell fails.**
- This project is managed by [uv](https://docs.astral.sh/uv/). From `code/mech-interp-codebase/`, run `uv sync` once to materialise `.venv` from `uv.lock`, then `uv run jupyter lab notebooks/` (or point your editor at `.venv/bin/python`).
- Gemma-2-2B is a gated HuggingFace model. Before first model-load: visit https://huggingface.co/google/gemma-2-2b, accept the licence, then `uv run huggingface-cli login` with a token that has access.

## 1. Imports + env metadata

In [ ]:
import importlib.metadata as md
import platform
import sys

import matplotlib
import numpy as np
import seaborn as sns
import torch
import transformer_lens

import mech_interp

# transformer_lens does not expose `__version__` as a module attribute
# (verified on 3.2.1) — read the installed dist version instead.
print(f'python           : {sys.version.split()[0]}')
print(f'platform         : {platform.platform()}')
print(f'torch            : {torch.__version__}')
print(f"transformer_lens : {md.version('transformer-lens')}")
print(f'numpy            : {np.__version__}')
print(f'matplotlib       : {matplotlib.__version__}')
print(f'seaborn          : {sns.__version__}')
print(f'mech_interp      : (editable from src/)')
print(f'cuda available   : {torch.cuda.is_available()}')
print(f'mps available    : {torch.backends.mps.is_available()}')

## 2. Device selection

Gemma-2-2B is 2B params → ~4–5 GB in fp32, ~2–3 GB in fp16/bf16. `mech_interp.select_device_dtype()` picks the best available device and a half-precision dtype where supported (bf16 on CUDA, fp16 on MPS, fp32 on CPU). MPS bf16 has known correctness issues in TransformerLens 3.x, so we use fp16 there.

In [ ]:
DEVICE, DTYPE = mech_interp.select_device_dtype()
print(f'device : {DEVICE}')
print(f'dtype  : {DTYPE}')

## 3. Load Gemma-2-2B

`mech_interp.get_model()` wraps `HookedTransformer.from_pretrained` with an `lru_cache(maxsize=1)`, so calling it twice in the same kernel returns the *same* in-memory model — no duplicate weights. The first call costs the full model download (~5 GB); subsequent calls hit the HF cache (`~/.cache/huggingface/`). Across kernels there is no sharing — see the module docstring in `src/mech_interp/__init__.py`.

In [ ]:
# Ref: A Mathematical Framework for Transformer Circuits — zotero://select/library/items/53EFBS8T
# See: vault/03-areas/research/mech-interp-overview.md ## Mechanism
MODEL_NAME = 'gemma-2-2b'

model = mech_interp.get_model(MODEL_NAME, device=DEVICE, dtype=DTYPE)

print(f'n_layers : {model.cfg.n_layers}')
print(f'd_model  : {model.cfg.d_model}')
print(f'n_heads  : {model.cfg.n_heads}')
print(f'd_head   : {model.cfg.d_head}')
print(f'd_vocab  : {model.cfg.d_vocab}')

## 4. Trivial forward pass

Gemma-2-2B is a *base* model (not instruction-tuned), so open-ended prompts like *'The capital of France is'* draw common pretraining continuations (`' a'`, `' the'`, …) ahead of the factual completion. A two-shot Q/A prompt anchors the format and makes *' Paris'* the unambiguous top-1. If the model loaded correctly, argmax on the last position is *' Paris'*; anything else (or NaN logits) means the load failed silently and Phase 1's residual-stream work cannot rely on this model.

In [ ]:
prompt = """Q: What is the capital of Germany?
A: Berlin

Q: What is the capital of Japan?
A: Tokyo

Q: What is the capital of France?
A:"""
tokens = model.to_tokens(prompt)
logits = model(tokens)

next_token_logits = logits[0, -1, :]
top5 = torch.topk(next_token_logits, k=5)

print(f'prompt: {prompt!r}')
print('top-5 next-token candidates:')
for prob, idx in zip(torch.softmax(top5.values, dim=-1).tolist(), top5.indices.tolist()):
    print(f'  {prob:.3f}  {model.to_string([idx])!r}')

argmax_token = model.to_string([next_token_logits.argmax().item()])
assert 'Paris' in argmax_token, f'expected Paris, got {argmax_token!r}'
print('\n✓ sanity check passed')

## 5. Residual-stream hook namespace

TransformerLens names hooks by site (e.g. `blocks.5.hook_resid_pre`). `01_residual_stream.ipynb` will filter on the `resid_pre` family — verify those hooks exist and the count matches `n_layers`.

In [ ]:
all_hooks = list(model.hook_dict.keys())
resid_pre_hooks = [h for h in all_hooks if h.endswith('hook_resid_pre')]

print(f'total hooks      : {len(all_hooks)}')
print(f'resid_pre hooks  : {len(resid_pre_hooks)}  (expected: n_layers = {model.cfg.n_layers})')
print('first three:', resid_pre_hooks[:3])

assert len(resid_pre_hooks) == model.cfg.n_layers, 'missing resid_pre hooks'
print('\n✓ hook namespace healthy')

## 6. Teardown

With fp16 on MPS, Gemma-2-2B weights are ~5 GB resident, plus a transient duplicate during TransformerLens's HF-weight rewrite. `mech_interp.unload_model()` clears the `lru_cache`, runs GC, and empties the device cache. To fully reclaim RAM, also shut the kernel down (VS Code: kernel picker → *Shut down kernel*) — if you are about to open `01_residual_stream.ipynb` in a separate kernel, do this first so you do not pay for two copies of the model at once.

In [ ]:
del model
mech_interp.unload_model()
print('model unloaded')

## Done

If all five cells executed cleanly, the env is ready for `01_residual_stream.ipynb`. Open observations from this run go in `## Log` of `vault/01-projects/code/mech-interp-codebase.md` — especially if the device/dtype choice deviated from the spec's CPU-local assumption.